# S1-01: API 설정과 첫 번째 요청
**Skilljar L03-L05: Accessing the API / Getting an API Key / Making a Request**

## 학습 목표
- Anthropic Python SDK 설치 및 환경 설정
- `.env` 파일을 통한 API 키 관리
- `client.messages.create()` 로 첫 API 호출 수행
- 응답 객체 구조 (content, usage, stop_reason) 분석

## 사전 준비
1. [Anthropic Console](https://console.anthropic.com)에서 API 키 발급
2. 이 노트북과 같은 폴더에 `.env` 파일 생성:
```
ANTHROPIC_API_KEY="sk-ant-api03-your-key-here"
```

In [ ]:
# 패키지 설치
%pip install anthropic python-dotenv

In [ ]:
# 환경변수 로드 (.env 파일에서 ANTHROPIC_API_KEY 읽기)
from dotenv import load_dotenv

load_dotenv()

In [ ]:
# Anthropic 클라이언트 생성
# Anthropic()은 환경변수 ANTHROPIC_API_KEY를 자동으로 인식한다
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-0"

## 1. 첫 번째 API 호출

`client.messages.create()`의 3가지 필수 매개변수:
- `model`: 사용할 모델명
- `max_tokens`: 최대 응답 토큰 수 (안전 한도)
- `messages`: 대화 메시지 리스트 (`role` + `content`)

In [ ]:
# 첫 API 호출 — 간단한 질문
message = client.messages.create(
    model=model,
    max_tokens=1024,
    messages=[
        {"role": "user", "content": "Claude API에 대해 한 문장으로 설명해주세요."}
    ]
)

# 응답 텍스트 출력
print(message.content[0].text)

## 2. 응답 객체 구조 분석

응답 객체에는 텍스트 외에도 유용한 메타데이터가 포함되어 있다:
- `role`: 항상 `"assistant"`
- `content[0].text`: 실제 응답 텍스트
- `stop_reason`: 응답 종료 사유 (`"end_turn"` | `"max_tokens"` | `"stop_sequence"`)
- `usage`: 토큰 사용량 (입력/출력)

In [ ]:
# 응답 객체의 주요 속성 확인
print(f"역할: {message.role}")                    # "assistant"
print(f"응답: {message.content[0].text}")          # 실제 응답 텍스트
print(f"중단 이유: {message.stop_reason}")          # "end_turn" (자연 종료)
print(f"입력 토큰: {message.usage.input_tokens}")   # 요청에 사용된 토큰
print(f"출력 토큰: {message.usage.output_tokens}")  # 응답에 사용된 토큰

In [ ]:
# 전체 응답 객체를 직접 확인
# content는 ContentBlock 리스트이므로, 멀티모달 응답도 지원한다
print(message)

## 3. max_tokens의 동작 이해

`max_tokens`는 "이만큼 생성하라"는 목표가 아니라 "이 이상은 절대 생성하지 마라"는 **안전 한도**이다.

- `stop_reason == "end_turn"`: 자연스럽게 응답 완료
- `stop_reason == "max_tokens"`: 토큰 한도에 도달하여 잘림 (응답이 불완전할 수 있음)

In [ ]:
# max_tokens를 매우 작게 설정하여 잘림 현상 관찰
short_message = client.messages.create(
    model=model,
    max_tokens=20,  # 매우 작은 토큰 한도
    messages=[
        {"role": "user", "content": "RC 보의 설계 절차를 상세하게 설명해주세요."}
    ]
)

print(f"응답: {short_message.content[0].text}")
print(f"중단 이유: {short_message.stop_reason}")  # "max_tokens" — 잘림!
print(f"출력 토큰: {short_message.usage.output_tokens}")

## 4. 여러 질문으로 실험

다양한 질문을 던져보면서 API 호출 패턴을 익힌다.

> **주의**: 각 `messages.create()` 호출은 **독립적**이다. 이전 호출의 맥락을 자동으로 기억하지 않는다.

In [ ]:
# 독립적인 두 번의 호출 — 서로 맥락을 공유하지 않음

# 첫 번째 호출
msg1 = client.messages.create(
    model=model,
    max_tokens=200,
    messages=[
        {"role": "user", "content": "콘크리트의 장점 3가지를 간단히 나열해줘."}
    ]
)
print("=== 호출 1 ===")
print(msg1.content[0].text)
print(f"(입력: {msg1.usage.input_tokens}, 출력: {msg1.usage.output_tokens} 토큰)")

# 두 번째 호출 — "아까 말한 것"을 모른다
msg2 = client.messages.create(
    model=model,
    max_tokens=200,
    messages=[
        {"role": "user", "content": "방금 말한 장점 중 첫 번째를 자세히 설명해줘."}
    ]
)
print("\n=== 호출 2 (맥락 없음) ===")
print(msg2.content[0].text)

---
## 건축공학 실습 과제

### 과제: RC 기둥 설계 검토 API 호출

아래 코드를 완성하여 RC 기둥의 설계 적정성을 Claude에게 검토 요청하세요.

**설계 조건:**
- 기둥 단면: 500mm x 500mm
- 콘크리트 강도 (fck): 24 MPa
- 철근 항복강도 (fy): 400 MPa
- 주근: 8-D25 (SD400)
- 설계 축력 (Pu): 2,500 kN
- 설계 모멘트 (Mu): 150 kN-m
- 적용 기준: KDS 14 20 20

**요구사항:**
1. 위 설계 조건을 포함한 프롬프트를 작성하세요
2. API를 호출하여 응답을 받으세요
3. 응답 텍스트와 토큰 사용량을 출력하세요

In [ ]:
# TODO: 아래 코드를 완성하세요

# RC 기둥 설계 검토 요청
column_review = client.messages.create(
    model=model,
    max_tokens=2048,
    messages=[
        {
            "role": "user",
            "content": """다음 RC 기둥의 설계 적정성을 검토해주세요.

설계 조건:
- 기둥 단면: 500mm x 500mm
- 콘크리트 강도 (fck): 24 MPa
- 철근 항복강도 (fy): 400 MPa
- 주근: 8-D25 (SD400)
- 띠철근: D10@300
- 설계 축력 (Pu): 2,500 kN
- 설계 모멘트 (Mu): 150 kN-m
- 적용 기준: KDS 14 20 20

검토 항목:
1. 축력비 검토 (최대 축력비 0.8 이하)
2. 최소/최대 철근비 검토 (0.01 <= rho <= 0.08)
3. 띠철근 간격 적정성 (KDS 14 20 22)
4. 판정 결과를 표 형식으로 정리"""
        }
    ]
)

# 응답 출력
print(column_review.content[0].text)
print(f"\n--- 토큰 사용량 ---")
print(f"입력: {column_review.usage.input_tokens} 토큰")
print(f"출력: {column_review.usage.output_tokens} 토큰")